In [9]:
import math
class Circle:
    def __init__(self,radius):
        self.radius = radius
    
    @property
    def area(self):
        return math.pi * (self.radius ** 2)
    
    @property
    def perimeter(self):
        return math.pi * self.radius * 2
    
c = Circle(10)
print(c.radius)
print(f'{c.area:.2f}')
print(f'{c.perimeter:.2f}')

10
314.16
62.83


In [17]:
class Temperature:
    def __init__(self,celsius):
        self.celsius = celsius
    
    @property
    def celsius(self):
        return self._celsius
    
    @celsius.setter
    def celsius(self,value):
        if value < -273.15:
            raise ValueError("温度不能低于绝对零度 (-273.15℃)！")
        self._celsius = value
    
    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32
    
    @fahrenheit.setter
    def fahrenheit(self,value):
        self.celsius = (value - 32) * 5/9

t = Temperature(50)
print(t.celsius)
print(t.fahrenheit)
t.fahrenheit = 32
print(t.celsius,t.fahrenheit)

50
122.0
0.0 32.0


In [27]:
import sys
class RegularParticle:
    def __init__(self,x,y,z):
        self.x = x
        self.y = y
        self.z = z
class SlottedParticle:
    __slots__ = ['x','y','z']
    def __init__(self,x,y,z):
        self.x = x
        self.y = y
        self.z = z

p1 = RegularParticle(1.0,2.0,3.0)
p2 = SlottedParticle(1.0,2.0,3.0)

print(f'普通对象内存:{sys.getsizeof(p1) + sys.getsizeof(p1.__dict__)}bytes')
print(f'slots对象内存:{sys.getsizeof(p2)}bytes')

p1.w = 4.0
print(f"普通对象新增属性 w: {p1.w}")
p2.w = 4.0

普通对象内存:352bytes
slots对象内存:56bytes
普通对象新增属性 w: 4.0


AttributeError: 'SlottedParticle' object has no attribute 'w'

In [ ]:
class BaseNoSlots:
    pass
class ChildWithSlots(BaseNoSlots):
    __slots__= ('a',)

c1 = ChildWithSlots()
c1.a = 1
print(c1.a)
c1.b = 2
print(c1.b)
print(hasattr(c1,'__dict__'))

1
2
True


In [34]:
class BaseWithSlots:
    __slots__ = ('x',)
class ChildBothSlots(BaseWithSlots):
    __slots__ = ('y',)

c2 = ChildBothSlots()
c2.x = 10
c2.y = 200
print(c2.x,c2.y)
print(hasattr(c2,'__dict__'))

10 200
False


In [38]:
class DictWrapper:
    def __init__(self,data_dict):
        self._data = data_dict
    def __getattr__(self, name):
        if name in self._data:
            return self._data[name]
        
        print(f"[警告] 属性 '{name}' 不存在，返回默认值 0")
        return 0

config = DictWrapper({'host':'loaclhost','port':8080})
print(config.host)
print(config.port)
print(config.timeout)

loaclhost
8080
[警告] 属性 'timeout' 不存在，返回默认值 0
0


In [27]:
class AccessLogger:
    def __init__(self,x,y):
        object.__setattr__(self,'_x',x)
        object.__setattr__(self,'_y',y)
        object.__setattr__(self,'_access_count',{})
    
    @property
    def x(self):
        return self._x
    
    @property
    def y(self):
        return self._y
    
    def __getattribute__(self,name):
        if not name.startswith('_'):
            counts = super().__getattribute__('_access_count')
            counts[name] = counts.get(name,0) + 1
            print(f'🔍 访问属性: {name} (第 {counts[name]} 次)')
        return super().__getattribute__(name)
    
obj = AccessLogger(10,20)
print(obj.x)
print(obj.y)
print(obj.x,obj.x)
print(f'\n统计: {obj._access_count}')
    

🔍 访问属性: x (第 1 次)
10
🔍 访问属性: y (第 1 次)
20
🔍 访问属性: x (第 2 次)
🔍 访问属性: x (第 3 次)
10 10

统计: {'x': 3, 'y': 1}


In [30]:
class TypedField:
    def __init__(self,expected_type):
        self.expected_type = expected_type
    def __set_name__(self,owner,name):
        self.public_name = name
        self.private_name = '_' + name
    def __get__(self,obj,objtype=None):
        if obj is None:
            return self
        return getattr(obj,self.private_name,None)  
    def __set__(self,obj,value):
        if not isinstance(value,self.expected_type):
            raise TypeError(f"'{self.public_name}' 必须是 {self.expected_type.__name__} 类型！")
        setattr(obj,self.private_name,value)
class User:
    name = TypedField(str)
    age = TypedField(int)
    height = TypedField(float)

    def __init__(self,name,age,height):
        self.name = name
        self.age = age
        self.height = height

u = User('Alice',25,1.68)
print(f'{u.name},{u.age}岁,{u.height}m')


                                


Alice,25岁,1.68m


In [34]:
import time
class LazyProperty:
    def __init__(self,func):
        self.func = func
        self.name = func.__name__
    def __get__(self,obj,objtype=None):
        if obj is None:
            return self
        
        print(f"⏳ 正在首次计算 '{self.name}' (耗时操作)...")
        value = self.func(obj)

        setattr(obj,self.name,value)
        return value
    
class DataAnalyzer:
    def __init__(self,data_size):
        self.data_size = data_size
    
    @LazyProperty
    def complex_result(self):
        time.sleep(2)
        return self.data_size * 42

analyzer = DataAnalyzer(1000)

print("开始第一次访问...")
start = time.time()
r1 = analyzer.complex_result
print(f"结果: {r1}, 耗时: {time.time() - start:.2f}s")

print("\n开始第二次访问...")
start = time.time()
r2 = analyzer.complex_result
print(f"结果: {r2}, 耗时: {time.time() - start:.2f}s")


    

开始第一次访问...
⏳ 正在首次计算 'complex_result' (耗时操作)...
结果: 42000, 耗时: 2.01s

开始第二次访问...
结果: 42000, 耗时: 0.00s


In [38]:
class ImmutablePoint:
    __slots__ = ('_x','_y','_initialized')
    def __init__(self,x,y):
        object.__setattr__(self,'_x',x)
        object.__setattr__(self,'_y',y)
        object.__setattr__(self,'_initialized',True)

    @property
    def x(self): return self._x

    @property
    def y(self): return self._y

    def __setattr__(self,name,value):
        if getattr(self,'_initialized',True):
            raise AttributeError(f"ImmutablePoint 对象是只读的，无法修改属性 '{name}'")
        object.__setattr__(self,name,value)
    def __repr__(self):
        return f'Point({self.x}, {self.y})'
p = ImmutablePoint(10,20)
print(p)
p.x = 30

Point(10, 20)


AttributeError: ImmutablePoint 对象是只读的，无法修改属性 'x'